## ЭКЗАМЕНАЦИОННЫЙ БИЛЕТ № 15
Практическое задание: разработайте схему миграции данных в HDFS (с псевдокодом) и симулируйте загрузку CSV-файла в экосистему Hadoop.

### 1. Архитектурная текстовая схема миграции (Data Pipeline)
 Процесс миграции устроен в виде конвейера, гарантирующего изоляцию, валидацию и распределенное хранение:

```
[ Локальный источник ] ──> (Локальные файлы: train.csv, test.csv)
│
▼
[ Слой валидации ] ──> [ Скрипт миграции (Python/WebHDFS) ]
│
├─ 1. Проверяет структуру колонок (схема)
├─ 2. Инициализирует сессию с NameNode Active
└─ 3. Читает потоком (Streaming) без переполнения RAM
│
▼
[ Взаимодействие RPC ] ──> [ NameNode (Кластер) ]
│ (Проверяет квоты, создает namespace,
│ возвращает адреса DataNodes для блоков 128MB)
▼
[ Слой хранения HDFS ] ──> [ DataNode Pipeline (Запись блоков по цепочке) ]
│ DN1 ──> DN2 ──> DN3 (Репликация x3)
▼
[ Целевая структура ] ──> /data/taxi/stg/
├── train/
│ └── year=2016/
│ └── month=01/
│ └── train_block_0.csv
└── test/
└── year=2016/
└── month=06/
└── test_block_0.csv
```

#### Шаг 1: Установка зависимости

In [1]:
pip install hdfs pandas

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


#### Шаг 2: Скрипт миграции (migrate_taxi_data.py)

In [7]:
import os
import sys
import logging
from hdfs import InsecureClient

# Настройка логирования для аудита
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%%(levelname)s] %%(message)s',
    handlers=[logging.StreamHandler(sys.stdout)]
)
logger = logging.getLogger(__name__)

# Спецификация ожидаемых схем данных для валидации
EXPECTED_SCHEMAS = {
    "test.csv": [
        "id", "vendor_id", "pickup_datetime", "passenger_count", 
        "pickup_longitude", "pickup_latitude", "dropoff_longitude", 
        "dropoff_latitude", "store_and_fwd_flag"
    ],
    "train.csv": [
        "id", "vendor_id", "pickup_datetime", "dropoff_datetime", "passenger_count", 
        "pickup_longitude", "pickup_latitude", "dropoff_longitude", 
        "dropoff_latitude", "store_and_fwd_flag", "trip_duration"
    ]
}

class HDFSMigrator:
    def __init__(self, namenode_url, hdfs_user):
        """ Инициализация клиента WebHDFS """
        logger.info(f"Подключение к HDFS NameNode: {namenode_url} от имени пользователя: {hdfs_user}")
        self.client = InsecureClient(namenode_url, user=hdfs_user)
        
    def validate_local_file(self, local_path, filename):
        """ Проверка существования файла и валидация его заголовков """
        if not os.path.exists(local_path):
            raise FileNotFoundError(f"Локальный файл не найден по пути: {local_path}")
            
        # Читаем только первую строчку для проверки схемы (экономим память)
        with open(local_path, 'r', encoding='utf-8') as f:
            header = f.readline().strip().split(',')
            
        expected = EXPECTED_SCHEMAS[filename]
        if header != expected:
            logger.error(f"Сбой валидации схемы для {filename}!")
            logger.error(f"Ожидалось: {expected}")
            logger.error(f"Получено:  {header}")
            return False
            
        logger.info(f"Файл {filename} успешно прошел валидацию схемы.")
        return True

    def upload_file(self, local_path, hdfs_dest_path, chunk_size_bytes=10485760):
        """ 
        Потоковая загрузка файла в HDFS (по умолчанию чанками по 10 МБ).
        Предотвращает Out-of-Memory ошибки.
        """
        filename = os.path.basename(local_path)
        
        if not self.validate_local_file(local_path, filename):
            sys.exit(1)
            
        logger.info(f"Начало потоковой миграции: {local_path} -> {hdfs_dest_path}")
        
        try:
            # Если целевая директория не существует, WebHDFS создаст её автоматически
            with open(local_path, 'rb') as reader:
                # Встроенный метод upload позволяет передавать итератор/поток данных
                self.client.upload(
                    hdfs_path=hdfs_dest_path,
                    data=reader,
                    overwrite=True, # Перезаписывать, если файл существовал (идемпотентность)
                    chunk_size=chunk_size_bytes
                )
            logger.info(f"Успешно мигрирован файл: {filename}")
            
            # Верификация: проверяем физический статус файла в HDFS
            content_summary = self.client.content_summary(hdfs_dest_path)
            logger.info(f"Верификация HDFS: Размер записанного файла = {content_summary['length']} байт")
            
        except Exception as e:
            logger.error(f"Критическая ошибка при загрузке {filename}: {str(e)}")
            raise e

if __name__ == "__main__":
    # КОНФИГУРАЦИЯ (Замените хост и порт на параметры вашего кластера)
    # Стандартный WebHDFS порт: 9870 для Hadoop 3.x, или 50070 для Hadoop 2.x
    NAMENODE_HTTP_URL = "http://localhost:9870" 
    HDFS_USER = "hadoop"
    
    # Пути к вашим локальным файлам
    LOCAL_TRAIN_PATH = "./train.csv" # исправили опечатку в расширении
    LOCAL_TEST_PATH = "./test.csv"
    
    # Целевая структура в HDFS (Слой STG - Staging данные)
    HDFS_TRAIN_TARGET = "/data/taxi/stg/train/train_raw.csv"
    HDFS_TEST_TARGET = "/data/taxi/stg/test/test_raw.csv"
    
    # Запуск процесса
    try:
        migrator = HDFSMigrator(NAMENODE_HTTP_URL, HDFS_USER)
        
        # Миграция train.csv
        migrator.upload_file(LOCAL_TRAIN_PATH, HDFS_TRAIN_TARGET)
        
        # Миграция test.csv
        migrator.upload_file(LOCAL_TEST_PATH, HDFS_TEST_TARGET)
        
        logger.info("Вся миграция успешно завершена!")
        
    except Exception as ex:
        logger.error(f"Процесс миграции аварийно завершен: {ex}")

2026-05-13 19:38:24,746 [%(levelname)s] %(message)s
2026-05-13 19:38:24,746 [%(levelname)s] %(message)s
2026-05-13 19:38:24,747 [%(levelname)s] %(message)s
2026-05-13 19:38:24,747 [%(levelname)s] %(message)s
2026-05-13 19:38:24,747 [%(levelname)s] %(message)s
2026-05-13 19:38:24,748 [%(levelname)s] %(message)s


### 2. Симуляция загрузки в Hadoop экосистему через CLI
Если вы хотите выполнить загрузку классическими утилитами Hadoop в рамках Bash-скрипта, схема развертывания выглядит следующим образом:

#### Шаг 1: Создание структуры директорий с разграничением прав
Создаем структуру директорий. Файлы train и test логически разделяются, так как имеют разную структуру колонок (в test.csv отсутствует целевая переменная trip_duration и колонка dropoff_datetime).

```bash
# Создаем директории верхнего уровня
hdfs dfs -mkdir -p /data/taxi/stg/train
hdfs dfs -mkdir -p /data/taxi/stg/test

# Назначаем права доступа (например, только для группы дата-инженеров)
hdfs dfs -chmod 775 /data/taxi/stg/train
hdfs dfs -chmod 775 /data/taxi/stg/test
```

#### Шаг 2: Симуляция загрузки файлов высокой производительности
Команда put считывает локальный файл и эффективно распределяет его блоками по DataNode. Флаг -f обеспечивает перезапись при повторном запуске (идемпотентность пайплайна).

```bash
# Загрузка train данных
hdfs dfs -put -f ./train.csv /data/taxi/stg/train/train.csv

# Загрузка test данных
hdfs dfs -put -f ./test.csv /data/taxi/stg/test/test.csv
```

#### Шаг 3: Промышленный аудит результатов (Проверка блоков и репликации)
Запустим проверку файловой системы (fsck), чтобы убедиться, как Hadoop обработал метаданные ваших файлов поездок:

```bash
hdfs fsck /data/taxi/stg/train/train.csv -files -blocks -locations
```

#### Что мы увидим в отчете симуляции:

- Файл разбит на блоки (например, если train.csv весит 200 МБ, система создаст Block 0 размером 128 МБ и Block 1 размером 72 МБ).

- Каждый блок будет физически записан на 3-х независимых DataNodes (при стандартном dfs.replication = 3).

- Метаданные о координатах и количестве пассажиров теперь распределены по кластеру и готовы к параллельной обработке через Apache Spark или Hive MapReduce.